# Transcription Evaluation Toolkit

Assumptions:
- Classical music with a reference score available in some cases
- Multiple transcriptions/performances of the same piece, though performances may differ from each other (e.g. repeated sections, cuts, added ornaments)

---
# Step 1 Alignment

In [3]:
import sys
sys.path.append('..')

import os
from pathlib import Path
from tqdm import tqdm
import csv
import time

import json
from itertools import combinations
from functools import partial

import numpy as np
import partitura as pt

import parangonar

from paths import *

from mpteval.preprocessing import chordify_perf_note_array
# alignment
from mpteval.alignment.dtw_align import process_pair_wrapper, align_chordified
from mpteval.alignment.dtw_dist import composite_cost

# import mpteval.alignment.dtw_align as dtw_align
# utils
from mpteval.utils import MusicEncoder

from concurrent.futures import ProcessPoolExecutor, as_completed

# Paths and Test Data

In [4]:
TEST_PIECES_PATH = data_path / 'TEST'

composer, cid = 'Schumann', '1515'
composer_id_folder = TEST_PIECES_PATH / composer / cid

cid_group = ATEPP_meta[ATEPP_meta["composition_id"] == int(cid)]
transcriptions = cid_group['perf_id'].unique()

pairs = list(combinations([i for i in transcriptions], 2))
print('First pair', pairs[0])
print(f"Generated {len(pairs):,} pairs from {len(transcriptions):,} transcriptions")

First pair ('10519', '10520')
Generated 21 pairs from 7 transcriptions


In [5]:
test = True

# select a pair
pair = pairs[0]
p1id, p2id = pair[0], pair[1]

p1p = ATEPP_root / cid_group[cid_group['perf_id'] == p1id]['norm_midi_path']


# Workflow for a single pair of performances

In [6]:
TEST_RUN = True

# select a pair
pair = pairs[0]
p1id, p2id = pair[0], pair[1]

p1p = str(ATEPP_transcriptions / cid_group[cid_group['perf_id'] == p1id]['norm_midi_path'].values[0])
p2p = str(ATEPP_transcriptions / cid_group[cid_group['perf_id'] == p2id]['norm_midi_path'].values[0])

# save to
out_dir = composer_id_folder / '0_test_align'
os.makedirs(out_dir, exist_ok=True)

if TEST_RUN:
    # 1. preprocess into chordified arrays
    p1 = pt.load_performance_midi(str(p1p)).performedparts[0].note_array()
    p2 = pt.load_performance_midi(str(p2p)).performedparts[0].note_array()
    p1c = chordify_perf_note_array(p1, ioi_threshold=0.03, max_threshold=0.5, return_list_of_dicts=True)
    p2c = chordify_perf_note_array(p2, ioi_threshold=0.03, max_threshold=0.5, return_list_of_dicts=True)

    # 2. align chords
    start = time.perf_counter()
    p1id, p2id, cost, align_len, p1_len, p2_len = align_chordified(p1c, p2c, dist_metric=composite_cost, out_dir = out_dir, p1id=p1id, p2id=p2id)
    print(f'Aligning {p1id}, {p2id}: cost={cost:.2f}, align_len={align_len}, p1_len={1480}, p2_len={p2_len}')
    print(f"Execution time: {time.perf_counter() - start:.4f}s")

# import shutil
# shutil.rmtree(out_dir)

Aligning 10519, 10520: cost=262.50, align_len=3848, p1_len=1480, p2_len=3550
Execution time: 111.1837s


# Parallelized workflow

In [7]:
# process all midis to chord lists

out_dir = composer_id_folder / '1_performance_data'
os.makedirs(out_dir, exist_ok=True)

pid_chord_list_dict = dict() 
for t in transcriptions:
    
    midi_path = str(ATEPP_transcriptions / cid_group[cid_group['perf_id'] == t]['norm_midi_path'].values[0])
    
    p1 = pt.load_performance_midi(midi_path).performedparts[0].note_array()
    # save performance note array
    np.save(f'{out_dir}/{t}.npy', p1)
    # add chordified list to dict
    p1c = chordify_perf_note_array(p1, ioi_threshold=0.03, max_threshold=0.5, return_list_of_dicts=True)
    pid_chord_list_dict[str(t)] = p1c

In [8]:
pid = list(pid_chord_list_dict.keys())[0]
pid_chord_list_dict[pid][:3] # first 3 chords

[{'pc_set': (9,),
  'pitch_set': (69,),
  'top_pitch': 69,
  'chord_onset': 1.8802084,
  'chord_onset_norm': 0.003990174,
  'nids': ['n0']},
 {'pc_set': (9,),
  'pitch_set': (69,),
  'top_pitch': 69,
  'chord_onset': 2.139323,
  'chord_onset_norm': 0.0045400662,
  'nids': ['n1']},
 {'pc_set': (10,),
  'pitch_set': (70,),
  'top_pitch': 70,
  'chord_onset': 2.6028645,
  'chord_onset_norm': 0.005523793,
  'nids': ['n2']}]

In [9]:
# save to file
if not os.path.exists(f"{out_dir}/pid_to_chords.json"):
    with open(f"{out_dir}/pid_to_chords.json", "w+", encoding="utf-8") as f:
        json.dump(pid_chord_list_dict, f, cls=MusicEncoder, indent=2)
else:
    with open(f"{out_dir}/pid_to_chords.json", "r", encoding="utf-8") as f:
        pid_to_chords_dict = json.load(f)

In [10]:
out_dir = composer_id_folder / '2_alignments'
os.makedirs(out_dir, exist_ok=True)

dist_metric = partial(composite_cost, alpha=0.85, harmonic_metric='jaccard', pitch_feature='pc_set')

process_func = partial(
process_pair_wrapper,
pid_to_chords_dict=pid_chord_list_dict,
dist_metric=dist_metric,
out_dir=out_dir,
directional_weights=np.array([1, 2, 1])
)

results = []

with ProcessPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_func, pair): pair for pair in pairs}
    
    with tqdm(total=len(pairs), desc='Current pairs progress') as pbar:
        for future in as_completed(futures):
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                pair = futures[future]  # Identify which pair failed
                print(f"Error processing pair {pair}: {e}")
            finally:
                pbar.update(1)

Current pairs progress: 100%|██████████| 21/21 [00:00<00:00, 345.54it/s]

Error processing pair ('10519', '10520'): Can't pickle <function process_pair_wrapper at 0x7c1f2f686f20>: import of module 'mpteval.alignment.dtw_align' failed
Error processing pair ('10519', '10521'): Can't pickle <function process_pair_wrapper at 0x7c1f2f686f20>: import of module 'mpteval.alignment.dtw_align' failed
Error processing pair ('10519', '10522'): Can't pickle <function process_pair_wrapper at 0x7c1f2f686f20>: import of module 'mpteval.alignment.dtw_align' failed
Error processing pair ('10519', '10523'): Can't pickle <function process_pair_wrapper at 0x7c1f2f686f20>: import of module 'mpteval.alignment.dtw_align' failed
Error processing pair ('10519', '10524'): Can't pickle <function process_pair_wrapper at 0x7c1f2f686f20>: import of module 'mpteval.alignment.dtw_align' failed
Error processing pair ('10519', '10525'): Can't pickle <function process_pair_wrapper at 0x7c1f2f686f20>: import of module 'mpteval.alignment.dtw_align' failed
Error processing pair ('10520', '10521')

In [11]:
# save results to csv
align_out_csv = composer_id_folder / '2_alignments' / 'align.csv'

with open(align_out_csv, mode='w+', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['p1', 'p2', 'cost', 'align_len', 'p1_len', 'p2_len'])
    writer.writerows(results)